In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/07_rag/04_rag_retrieval.py

# Phase 14.5 — RAG Answer Generation

This notebook generates grounded answers using retrieved business documents.

Input:

- User question
- Retrieved document chunks
- RAG context

Output:

- Grounded answer
- Source citations
- Retrieved sources
- Retrieval scores

The LLM must answer only from the supplied context.
If the context does not contain enough information, the system must refuse to invent an answer.

In [0]:
import json
import time

from pyspark.sql import functions as F

print("Imports successful.")

In [0]:
LLM_MODEL = "databricks-meta-llama-3-3-70b-instruct"

TOP_K = 3

print("LLM model:", LLM_MODEL)
print("Top-K:", TOP_K)

In [0]:
question = "What is the discount policy?"

print("Question:")
print(question)

In [0]:
print("Question:", question)

print(
    "Retrieved chunks:",
    len(retrieval_results[:TOP_K])
)

print(
    "RAG context characters:",
    len(rag_context)
)

In [0]:
def build_rag_prompt(question, rag_context):
    """
    Build a grounded RAG prompt.

    The model is explicitly instructed to:
    - use only supplied context
    - avoid inventing information
    - cite sources
    - refuse unsupported questions
    """

    prompt = f"""
You are a business data assistant.

Answer the user's question using ONLY the document context
provided below.

IMPORTANT RULES:

1. Do not use outside knowledge.
2. Do not invent facts.
3. Do not invent policies.
4. If the context does not contain enough information,
   say that the available documents do not contain enough
   information to answer the question.
5. Keep the answer concise and professional.
6. Cite the source document used for the answer.
7. When multiple sources support the answer, cite each relevant source.

USER QUESTION:

{question}

DOCUMENT CONTEXT:

{rag_context}

RESPONSE FORMAT:

Answer:
<grounded answer>

Sources:
- <file name> — <title>
"""

    return prompt.strip()

In [0]:
rag_prompt = build_rag_prompt(
    question,
    rag_context
)

print(rag_prompt)

In [0]:
def generate_rag_answer(question, context):
    """
    Generate a grounded RAG answer using Databricks ai_query().
    """

    start_time = time.time()

    try:

        if not question:

            return {
                "success": False,
                "answer": None,
                "error": "Question is empty.",
                "latency_ms": 0.0
            }

        if not context:

            return {
                "success": False,
                "answer": None,
                "error": "RAG context is empty.",
                "latency_ms": 0.0
            }

        # --------------------------------------------------
        # 1. Build grounded prompt
        # --------------------------------------------------

        prompt = build_rag_prompt(
            question,
            context
        )

        # --------------------------------------------------
        # 2. Escape values for SQL
        # --------------------------------------------------

        safe_prompt = prompt.replace(
            "'",
            "''"
        )

        safe_model = LLM_MODEL.replace(
            "'",
            "''"
        )

        # --------------------------------------------------
        # 3. Call Databricks Foundation Model
        # --------------------------------------------------

        query = f"""
        SELECT ai_query(
            '{safe_model}',
            '{safe_prompt}'
        ) AS answer
        """

        result_df = spark.sql(
            query
        )

        result_row = result_df.first()

        if result_row is None:

            return {
                "success": False,
                "answer": None,
                "error": "LLM returned no result.",
                "latency_ms": round(
                    (time.time() - start_time) * 1000,
                    2
                )
            }

        answer = result_row["answer"]

        if answer is None or not str(answer).strip():

            return {
                "success": False,
                "answer": None,
                "error": "LLM returned an empty answer.",
                "latency_ms": round(
                    (time.time() - start_time) * 1000,
                    2
                )
            }

        return {
            "success": True,
            "answer": str(answer).strip(),
            "error": None,
            "latency_ms": round(
                (time.time() - start_time) * 1000,
                2
            )
        }

    except Exception as e:

        return {
            "success": False,
            "answer": None,
            "error": (
                f"{type(e).__name__}: {str(e)}"
            ),
            "latency_ms": round(
                (time.time() - start_time) * 1000,
                2
            )
        }

In [0]:
import inspect

print(
    inspect.getsource(
        generate_rag_answer
    )
)

In [0]:
rag_question = "What is the discount policy?"

question_embedding = generate_question_embedding(
    rag_question
)

retrieval_results = retrieve_documents(
    question_embedding,
    top_k=5
)

rag_context = build_rag_context(
    retrieval_results
)

rag_result = generate_rag_answer(
    rag_question,
    rag_context
)

print(
    json.dumps(
        rag_result,
        indent=2,
        default=str
    )
)

In [0]:
rag_result = generate_rag_answer(
    question,
    rag_context
)

print(
    json.dumps(
        rag_result,
        indent=2,
        default=str
    )
)

In [0]:
print("=" * 60)
print("RAG ANSWER")
print("=" * 60)

print(rag_result["answer"])

print("=" * 60)
print("Latency:", rag_result["latency_ms"], "ms")

In [0]:
def build_source_citations(retrieval_results, top_k=TOP_K):
    """
    Build source metadata from retrieved chunks.
    """

    citations = []

    for rank, item in enumerate(
        retrieval_results[:top_k],
        start=1
    ):
        citations.append(
            {
                "rank": rank,
                "file_name": item.get(
                    "file_name",
                    "Unknown"
                ),
                "title": item.get(
                    "title",
                    "Unknown"
                ),
                "chunk_id": item.get(
                    "chunk_id",
                    "Unknown"
                ),
                "similarity": float(
                    item.get(
                        "similarity",
                        0.0
                    )
                )
            }
        )

    return citations

In [0]:
citations = build_source_citations(
    retrieval_results
)

print(
    json.dumps(
        citations,
        indent=2,
        default=str
    )
)

In [0]:
final_rag_response = {
    "success": rag_result["success"],
    "question": question,
    "answer": rag_result["answer"],
    "sources": citations,
    "retrieved_chunks": len(
        retrieval_results[:TOP_K]
    ),
    "latency_ms": rag_result["latency_ms"]
}

print(
    json.dumps(
        final_rag_response,
        indent=2,
        default=str
    )
)

In [0]:
print("=" * 70)
print("AI DATA ANALYST COPILOT — RAG RESPONSE")
print("=" * 70)

print("\nQuestion:")
print(question)

print("\nAnswer:")
print(rag_result["answer"])

print("\nSources:")

for source in citations:
    print(
        f"- {source['file_name']} "
        f"| {source['title']} "
        f"| similarity={source['similarity']:.4f}"
    )

print(
    f"\nProcessing time: "
    f"{rag_result['latency_ms']} ms"
)

print("=" * 70)

In [0]:
question = "What is the company's employee vacation policy?"

In [0]:
question_embedding = generate_question_embedding(
    question
)

retrieval_results = []

for row in document_rows:

    similarity = cosine_similarity(
        question_embedding,
        row["embedding"]
    )

    retrieval_results.append(
        {
            "chunk_id": row["chunk_id"],
            "document_id": row["document_id"],
            "file_name": row["file_name"],
            "document_type": row["document_type"],
            "title": row["title"],
            "chunk_index": row["chunk_index"],
            "chunk_text": row["chunk_text"],
            "similarity": float(similarity)
        }
    )

retrieval_results = sorted(
    retrieval_results,
    key=lambda x: x["similarity"],
    reverse=True
)

rag_context = build_rag_context(
    retrieval_results
)

rag_result = generate_rag_answer(
    question,
    rag_context
)

print(rag_result["answer"])

In [0]:
question = "What is the discount policy?"

question_embedding = generate_question_embedding(
    question
)

retrieval_results = []

for row in document_rows:

    similarity = cosine_similarity(
        question_embedding,
        row["embedding"]
    )

    retrieval_results.append(
        {
            "chunk_id": row["chunk_id"],
            "document_id": row["document_id"],
            "file_name": row["file_name"],
            "document_type": row["document_type"],
            "title": row["title"],
            "chunk_index": row["chunk_index"],
            "chunk_text": row["chunk_text"],
            "similarity": float(similarity)
        }
    )

retrieval_results = sorted(
    retrieval_results,
    key=lambda x: x["similarity"],
    reverse=True
)

rag_context = build_rag_context(
    retrieval_results
)

rag_result = generate_rag_answer(
    question,
    rag_context
)

citations = build_source_citations(
    retrieval_results
)

print(rag_result["answer"])

print("\nSources:")

for source in citations:
    print(
        f"- {source['file_name']} "
        f"| {source['title']}"
    )

In [0]:
print("=" * 70)
print("PHASE 14.5 VALIDATION")
print("=" * 70)

checks = []

checks.append(
    (
        "Question exists",
        bool(question.strip())
    )
)

checks.append(
    (
        "Retrieved chunks exist",
        len(retrieval_results) > 0
    )
)

checks.append(
    (
        "RAG context exists",
        bool(rag_context.strip())
    )
)

checks.append(
    (
        "LLM answer exists",
        bool(
            rag_result.get("answer")
        )
    )
)

checks.append(
    (
        "Sources exist",
        len(citations) > 0
    )
)

for name, passed in checks:
    print(
        f"{'PASS' if passed else 'FAIL'} - {name}"
    )

failed = sum(
    1
    for _, passed in checks
    if not passed
)

print("\nTotal checks:", len(checks))
print("Failed checks:", failed)

if failed == 0:
    print("Overall status: PASS")
else:
    print("Overall status: FAIL")

In [0]:
print("=" * 70)
print("PHASE 14.5 — RAG ANSWER GENERATION")
print("=" * 70)

for name in [
    "generate_question_embedding",
    "retrieve_documents",
    "build_rag_context"
]:
    print(
        f"{'PASS' if callable(globals().get(name)) else 'FAIL'} - {name}"
    )

In [0]:
rag_question = "What is the discount policy?"

question_embedding = generate_question_embedding(
    rag_question
)

retrieval_results = retrieve_documents(
    question_embedding,
    top_k=5
)

rag_context = build_rag_context(
    retrieval_results
)

rag_result = generate_rag_answer(
    rag_question,
    rag_context
)

print(
    json.dumps(
        rag_result,
        indent=2,
        default=str
    )
)

In [0]:
print("=" * 70)
print("RAG ANSWER")
print("=" * 70)

if rag_result.get("success"):

    print(
        rag_result.get("answer")
    )

else:

    print(
        "RAG failed:",
        rag_result.get("error")
    )

In [0]:
print("=" * 70)
print("PHASE 14.5 CHECK")
print("=" * 70)

print(
    "generate_rag_answer:",
    "PASS"
    if callable(
        globals().get(
            "generate_rag_answer"
        )
    )
    else "FAIL"
)

05_rag_answer_generation

Go to your 05_rag_answer_generation file.

At the moment, it apparently does not contain the function you expected. Add the actual implementation there.

Since your project is using Databricks GenAI, first let's see what LLM/AI helper functions are already available in that notebook.

Run this inside 05_rag_answer_generation:


In [0]:
import inspect

print("=" * 70)
print("AVAILABLE AI / LLM FUNCTIONS")
print("=" * 70)

for name, obj in list(globals().items()):

    if callable(obj) and not name.startswith("_"):

        if (
            "ai" in name.lower()
            or "llm" in name.lower()
            or "genai" in name.lower()
            or "chat" in name.lower()
            or "model" in name.lower()
        ):
            print(name)

In [0]:
print("=" * 70)
print("AVAILABLE IMPORTS / OBJECTS")
print("=" * 70)

for name in sorted(
    [
        x for x in globals().keys()
        if not x.startswith("_")
    ]
):
    print(name)

In [0]:
print(
    type(LLM_MODEL)
)

print(
    LLM_MODEL
)

In [0]:
print(type(LLM_MODEL))
print(LLM_MODEL)

In [0]:
print(inspect.getsource(build_rag_prompt))

In [0]:
print("LLM_MODEL type:", type(LLM_MODEL))
print("LLM_MODEL:", LLM_MODEL)

In [0]:
import inspect

print("=" * 70)
print("LLM_MODEL METHODS")
print("=" * 70)

print(
    [
        name
        for name in dir(LLM_MODEL)
        if not name.startswith("_")
    ]
)

In [0]:
import inspect

print(
    inspect.getsource(
        generate_rag_answer
    )
)

test


In [0]:
import inspect


print(
    inspect.getsource(
        generate_rag_answer
    )
)